# Stronger-models overconfidence test

Does the African-language **overconfidence gap** generalize to a frontier model (`google/gemini-2.5-pro`) and a reasoning model (`deepseek/deepseek-r1-0528`)?

We use **verbalized confidence**: the model answers a multiple-choice question and states an integer confidence 0-100. This notebook validates every piece before the full run. It reuses the production functions in `run_inference.py`.

## 0. Setup

In [1]:
import os, sys, json
from types import SimpleNamespace
sys.path.insert(0, os.path.abspath('.'))
import run_inference as R

R.load_env()
assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing in .env'
API_KEY = os.environ['OPENROUTER_API_KEY']
print('key loaded, models:', R.DEFAULT_MODELS)

key loaded, models: ['google/gemini-2.5-pro', 'deepseek/deepseek-r1-0528']


## 1. Load and inspect the datasets
For speed we inspect English + two African languages here. The full run uses every available language (see `run_inference.py`).

In [2]:
# reduced configs for a fast walkthrough (eng + 2 African langs each)
uhura_demo = SimpleNamespace(**{**vars(R.UHURA), 'languages': ['eng', 'swa', 'yor']})
afri_demo  = SimpleNamespace(**{**vars(R.AFRIMMLU), 'languages': ['eng', 'swa', 'yor']})

uhura = R.load_benchmark(uhura_demo)
afri  = R.load_benchmark(afri_demo)
print('Uhura langs:', {l: len(d) for l, d in uhura.items()})
print('AfriMMLU langs:', {l: len(d) for l, d in afri.items()})

Uhura langs: {'eng': 809, 'swa': 807, 'yor': 809}
AfriMMLU langs: {'eng': 500, 'swa': 500, 'yor': 500}


In [3]:
# inspect one AfriMMLU item
idx0 = sorted(afri['eng'])[0]
s = afri['eng'][idx0]
labs = R.letters(len(s.choices))
print('question:', s.question)
for lab, c in zip(labs, s.choices):
    print(f'  {lab}. {c}')
print('gold:', labs[s.answer_index])

question: What is the value of p in 24 = 2p?
  A. p = 4
  B. p = 8
  C. p = 12
  D. p = 24
gold: C


## 2. Matched question samples
The same question indices are used across every language (fixed seed), so English vs African is paired.

In [4]:
afri_idx  = R.matched_indices(afri, n=5, seed=R.SEED)
uhura_idx = R.matched_indices(uhura, n=5, seed=R.SEED)
print('AfriMMLU matched idx:', afri_idx)
print('Uhura matched idx:', uhura_idx)

AfriMMLU matched idx: [12, 57, 140, 327, 379]
Uhura matched idx: [25, 114, 281, 654, 759]


## 3. The prompt (identical instruction, English and African)

In [5]:
p_en, _ = R.build_prompt(afri['eng'][afri_idx[0]])
p_af, _ = R.build_prompt(afri['swa'][afri_idx[0]])
print('--- English ---\n', p_en)
print('\n--- Swahili (same instruction) ---\n', p_af)

--- English ---
 Answer the following multiple-choice question.

Question: Maddie will ride her bike a total of 56 miles over 7 days. She will ride the same number of miles each day. What is the total number of miles Maddie will ride each day?

Options:
A. 8
B. 9
C. 49
D. 63

Respond ONLY with a JSON object of the form {"answer": "<letter>", "confidence": <integer 0-100>} where confidence is how sure you are that your answer is correct. Do not include any other text.

--- Swahili (same instruction) ---
 Answer the following multiple-choice question.

Question: Maddie ataendesha baiskeli yake jumla ya maili 56 kwa siku 7. Ataendesha idadi hii hii ya maili kila siku. Je, Maddie ataendesha baiskeli maili ngapi kila siku?

Options:
A. 8
B. 9
C. 49
D. 63

Respond ONLY with a JSON object of the form {"answer": "<letter>", "confidence": <integer 0-100>} where confidence is how sure you are that your answer is correct. Do not include any other text.


## 4. Test both OpenRouter models

In [6]:
sample = afri['eng'][afri_idx[0]]
prompt, labs = R.build_prompt(sample)
gold = labs[sample.answer_index]
for model in R.DEFAULT_MODELS:
    data, status, err, transient = R.call_model(model, prompt, API_KEY, timeout=180)
    raw = R.content_of(data) if data else ''
    ans, conf, perr = R.parse_response(raw)
    inp, out, rt, cost = R.usage_fields(data) if data else (0, 0, 0, 0.0)
    print(f'\n=== {model} (http {status}) ===')
    print('raw:', raw[:200])
    print(f'answer={ans} conf={conf} correct={ans==gold} | in={inp} out={out} reasoning={rt} cost=${cost:.5f} err={perr}')


=== google/gemini-2.5-pro (http 200) ===
raw: ```json
{
  "answer": "A",
  "confidence": 100
}
```
answer=A conf=100.0 correct=True | in=125 out=820 reasoning=795 cost=$0.00836 err=None

=== deepseek/deepseek-r1-0528 (http 200) ===
raw: {"answer": "A", "confidence": 100}
answer=A conf=100.0 correct=True | in=125 out=560 reasoning=547 cost=$0.00128 err=None


## 5. Verify answer / confidence parsing (including messy formats)

In [7]:
tests = [
    '{"answer": "B", "confidence": 82}',
    '```json\n{"answer": "C", "confidence": 100}\n```',
    'The answer is A.\n{"answer":"A","confidence":55}',
    'answer: D confidence: 40',
    'I cannot answer this.',
]
for t in tests:
    print(R.parse_response(t), '  <-', t[:40].replace(chr(10), ' '))

('B', 82.0, None)   <- {"answer": "B", "confidence": 82}
('C', 100.0, None)   <- ```json {"answer": "C", "confidence": 10
('A', 55.0, None)   <- The answer is A. {"answer":"A","confiden
(None, None, 'no_json_found')   <- answer: D confidence: 40
(None, None, 'no_json_found')   <- I cannot answer this.


## 6. Tiny end-to-end pilot
Run 2 matched questions across 3 languages (~12 calls, ~1-2 min) with a tight cost cap. This exercises the full resumable pipeline and writes to `outputs/results.jsonl`.

Note: `capture_output=True` buffers, so output appears only when the cell finishes. The pilot writes real rows; clear the file before the full run so pilot rows do not mix in:
```python
open('outputs/results.jsonl', 'w').close()
```

In [9]:
import subprocess
# Tiny pilot: 2 questions x 3 languages x 2 models = 12 calls (~1-2 min, a few cents).
out = subprocess.run(
    [sys.executable, 'run_inference.py', '--limit', '2', '--benchmarks', 'afrimmlu',
     '--languages', 'eng', 'swa', 'yor', '--cost-limit', '1.0'],
    capture_output=True, text=True)
print(out.stdout[-1500:])
print(out.stderr[-500:])

[data] loading afrimmlu ...
[data] afrimmlu: 3 languages, 2 matched questions (eng, swa, yor)
[state] 11 prior records, $0.08 spent, 11 completed keys
[plan] 12 evaluations (2 models x 6 items); cost limit $1.00


Done. Completed: 19/12   Failures: 0   Total cost: $0.14
Results: /Users/similoluwaokunowo/Desktop/Codes/Apart AI Safety Hackathon/experiments/stronger_models/outputs/results.jsonl




In [10]:
# inspect the saved records
path = os.path.join('outputs', 'results.jsonl')
recs = [json.loads(l) for l in open(path)] if os.path.exists(path) else []
print(f'{len(recs)} records; statuses:', {s: sum(r["status"]==s for r in recs) for s in {r["status"] for r in recs}})
for r in recs[:4]:
    print(f"  {r['model'][:22]:22s} {r['benchmark']} {r['language']} q{r['question_id']} "
          f"-> {r['parsed_answer']} conf={r['confidence']} correct={r['correct']} ${r['cost_usd']}")

19 records; statuses: {'success': 19}
  google/gemini-2.5-pro  afrimmlu eng q57 -> A conf=100.0 correct=True $0.006981
  deepseek/deepseek-r1-0 afrimmlu eng q57 -> A conf=100.0 correct=True $0.000958
  google/gemini-2.5-pro  afrimmlu eng q327 -> C conf=100.0 correct=True $0.00725
  deepseek/deepseek-r1-0 afrimmlu eng q327 -> C conf=90.0 correct=True $0.002371


## 7. Confirm, then run the full experiment
Everything above should work: data loads, prompts build, both models answer, parsing holds, records save and resume.

Full run (resumable; raise `--cost-limit` for full coverage, ~\$9-10 for 50 items x both benchmarks x both models):
```bash
uv run python run_inference.py --limit 50 --cost-limit 10
uv run python analyze_results.py
```